# Crop Prediction Model Evaluation

This notebook evaluates the performance of our crop prediction model using the dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

In [2]:
# Load the dataset
df = pd.read_csv('datasets/newdataset.csv', delimiter='\t')
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()

Dataset shape: (100, 17)

First few rows:


,field_id,land,temperature,humidity,rainfall,budget,soil_type,ph,water_requirement,suggested_crop,suggested_fertilizers,suggested_pesticides,potential_diseases,nitrogen,phosphorus,potassium,estimated_yield
0,1,10,25,60,500,5000,Sandy,6.5,20,Rice,Ammonium Nitrate,Malathion,Blast,130,65,95,5
1,2,15,28,55,600,7000,Clay,7.0,25,Wheat,Urea,Chlorpyrifos,Rust,120,60,90,7
2,3,20,22,65,450,6000,Loam,6.0,18,Corn,Ammonium Nitrate,Malathion,Maize Lethal Necrosis,150,75,100,6
3,4,12,27,58,550,6500,Sandy,6.8,22,Rice,DAP,Imidacloprid,Blast,130,65,95,6
4,5,18,24,62,480,6200,Clay,7.2,24,Soybean,Potassium Nitrate,Metomil,Asian Soybean Rust,140,70,105,8


## 2. Data Analysis

In [ ]:
# Check class distribution
plt.figure(figsize=(12, 6))
df['suggested_crop'].value_counts().plot(kind='bar')
plt.title('Distribution of Crop Types')
plt.xlabel('Crop Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [3]:
# Create advanced features
def create_advanced_features(df):
    df = df.copy()
    
    # NPK Interaction Score
    df['npk_interaction'] = (
        (df['nitrogen'] * df['phosphorus'] * df['potassium']) ** (1/3)
    )
    
    # Water Stress Index
    df['water_stress_index'] = (
        (df['rainfall'] / df['water_requirement']).clip(0, 2)
    )
    
    # Soil Health Score
    optimal_ph = 7.0
    df['soil_health_score'] = (
        10 - (df['ph'] - optimal_ph).abs() + 
        (df['nitrogen'] / 150) + 
        (df['phosphorus'] / 75) + 
        (df['potassium'] / 100)
    ).clip(0, 10)
    
    # Climate Suitability Index
    df['climate_index'] = (
        (df['temperature'] - 20).abs() / 10 + 
        (df['humidity'] - 60).abs() / 20
    ).clip(0, 1)
    
    return df

df_engineered = create_advanced_features(df)
print("New features created successfully")

New features created successfully


## 4. Model Training and Evaluation

In [4]:
# Prepare features and target
feature_columns = [
    'land', 'temperature', 'humidity', 'rainfall', 
    'soil_type', 'ph', 'nitrogen', 'phosphorus', 'potassium',
    'npk_interaction', 'water_stress_index', 
    'soil_health_score', 'climate_index'
]

# Encode categorical variables
soil_encoder = LabelEncoder()
df_engineered['soil_type'] = soil_encoder.fit_transform(df_engineered['soil_type'])

X = df_engineered[feature_columns]
y = df_engineered['suggested_crop']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE for handling class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Train model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_balanced, y_train_balanced)

# Make predictions
y_pred = knn.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

# Cross-validation score
cv_scores = cross_val_score(knn, X_train_balanced, y_train_balanced, cv=5)
print(f"\nCross-validation scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

## 5. Confusion Matrix Visualization

In [5]:
# Plot confusion matrix
plt.figure(figsize=(12, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

NameError: name 'y_test' is not defined

<Figure size 1200x800 with 0 Axes>

## 6. Feature Importance Analysis

In [ ]:
# Calculate feature importance based on correlation with predictions
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': np.abs(np.corrcoef(X_train_scaled.T, y_train_balanced.factorize()[0])[:-1, -1])
})
feature_importance = feature_importance.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance)
plt.title('Feature Importance')
plt.xlabel('Absolute Correlation with Target')
plt.tight_layout()
plt.show()